In [1]:
pip install tensorflow pandas numpy scikitFashion Trend Prediction using Myntra Dataset-learn matplotlib

#**1. IMPORT LIBRARIES**

In [2]:
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

#**2. LOAD DATASET**

In [3]:
df=pd.read_csv('/content/Fashion Dataset v2.csv',on_bad_lines='skip')

In [4]:
print(df.head())

       p_id  ...                                       p_attributes
0  17048614  ...  {'Add-Ons': 'NA', 'Body Shape ID': '443,333,32...
1  16524740  ...  {'Add-Ons': 'NA', 'Body Shape ID': '443,333,32...
2  16331376  ...  {'Add-Ons': 'NA', 'Body Shape ID': '333,424', ...
3  14709966  ...  {'Add-Ons': 'NA', 'Body Shape ID': '333,424', ...
4  11056154  ...  {'Body Shape ID': '424', 'Body or Garment Size...

[5 rows x 11 columns]


#**3. SELECT IMPORTANT COLUMNS**

In [11]:
columns = [
    "products",
    "colour",
    "brand",
    "description"
]

df = df[columns]

In [12]:
# REMOVE NULL VALUES
df.dropna(inplace=True)

/tmp/ipykernel_904/3599349860.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.dropna(inplace=True)


In [13]:
# CREATE SEQUENCE
df["sequence"] = (
    df["brand"].astype(str) + " " +
    df["colour"].astype(str) + " " +
    df["description"].astype(str)
)

/tmp/ipykernel_904/2259653703.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["sequence"] = (


In [14]:
# INPUT + OUTPUT
X = df["sequence"]

y = df["products"]


#**4. LABEL ENCODING**

In [15]:
encoder = LabelEncoder()

y = encoder.fit_transform(y)

#**5. TOKENIZATION**

In [16]:
tokenizer = Tokenizer(num_words=10000)

tokenizer.fit_on_texts(X)

X_seq = tokenizer.texts_to_sequences(X)

#**6. PADDING**

In [17]:
max_len = 15

X_pad = pad_sequences(
    X_seq,
    maxlen=max_len,
    padding='post'
)

#**7. TRAIN TEST SPLIT**

In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X_pad,
    y,
    test_size=0.2,
    random_state=42
)

#**8. VOCAB SIZE**

In [19]:
vocab_size = len(tokenizer.word_index) + 1

#**9. BUILD GRU MODEL**

In [23]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Embedding,
    GRU,
    Dense,
    Dropout
)

model = Sequential()

# Explicit Input Layer
model.add(Input(shape=(15,)))

# Embedding Layer
model.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=128
    )
)

# GRU Layer
model.add(
    GRU(128)
)

# Dropout
model.add(
    Dropout(0.3)
)

# Dense Layer
model.add(
    Dense(64, activation='relu')
)

# Output Layer
model.add(
    Dense(
        len(np.unique(y)),
        activation='softmax'
    )
)


#**10. COMPILE MODEL**

In [24]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

#**11. MODEL SUMMARY**

In [25]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 15, 128)        │       944,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 128)            │        99,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 910)            │        59,150 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,110,734 (4.24 MB)

 Trainable params: 1,110,734 (4.24 MB)

 Non-trainable params: 0 (0.00 B)

#**12. EARLY STOPPING**

In [26]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

#**13. TRAIN MODEL**

In [27]:
history = model.fit(
    X_train,
    y_train,
    epochs=15,
    batch_size=64,
    validation_data=(X_test, y_test),
    callbacks=[early_stop]
)

Epoch 1/15
178/178 ━━━━━━━━━━━━━━━━━━━━ 13s 57ms/step - accuracy: 0.0974 - loss: 4.5446 - val_accuracy: 0.1548 - val_loss: 3.9741
Epoch 2/15
178/178 ━━━━━━━━━━━━━━━━━━━━ 10s 55ms/step - accuracy: 0.1876 - loss: 3.6546 - val_accuracy: 0.2086 - val_loss: 3.5406
Epoch 3/15
178/178 ━━━━━━━━━━━━━━━━━━━━ 9s 49ms/step - accuracy: 0.2691 - loss: 3.2503 - val_accuracy: 0.3218 - val_loss: 3.2510
Epoch 4/15
178/178 ━━━━━━━━━━━━━━━━━━━━ 11s 62ms/step - accuracy: 0.3814 - loss: 2.8885 - val_accuracy: 0.4158 - val_loss: 2.9807
Epoch 5/15
178/178 ━━━━━━━━━━━━━━━━━━━━ 20s 58ms/step - accuracy: 0.4504 - loss: 2.5951 - val_accuracy: 0.4720 - val_loss: 2.8379
Epoch 6/15
178/178 ━━━━━━━━━━━━━━━━━━━━ 10s 57ms/step - accuracy: 0.4939 - loss: 2.3931 - val_accuracy: 0.4917 - val_loss: 2.7648
Epoch 7/15
178/178 ━━━━━━━━━━━━━━━━━━━━ 8s 46ms/step - accuracy: 0.5239 - loss: 2.2449 - val_accuracy: 0.5076 - val_loss: 2.7250
Epoch 8/15
178/178 ━━━━━━━━━━━━━━━━━━━━ 13s 72ms/step - accuracy: 0.5510 - loss: 2.1148 - va

#**14. EVALUATE MODEL**

In [28]:
loss, accuracy = model.evaluate(X_test, y_test)

print("\nTest Accuracy:", accuracy)

89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.5297 - loss: 2.6993

Test Accuracy: 0.5297221541404724


#**15. SAVE MODEL**

In [29]:
model.save("fashion_gru_model.keras")

pickle.dump(
    tokenizer,
    open("fashion_tokenizer.pkl", "wb")
)

pickle.dump(
    encoder,
    open("fashion_encoder.pkl", "wb")
)

print("\nModel Saved Successfully!")


Model Saved Successfully!


#**16. PREDICTION FUNCTION**

In [30]:
def predict_fashion(sequence):

    seq = tokenizer.texts_to_sequences([sequence])

    padded = pad_sequences(
        seq,
        maxlen=max_len,
        padding='post'
    )

    prediction = model.predict(padded)

    predicted_class = np.argmax(prediction)

    fashion_type = encoder.inverse_transform(
        [predicted_class]
    )[0]

    confidence = np.max(prediction) * 100

    print("\nInput:", sequence)

    print("Predicted Fashion Type:", fashion_type)

    print(f"Confidence: {confidence:.2f}%")

#**17. TEST PREDICTIONS**

In [31]:
predict_fashion(
    "Apparel Ethnic Wear Red Summer"
)

predict_fashion(
    "Apparel Topwear Black Winter"
)

predict_fashion(
    "Accessories Watches Blue Casual"
)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 274ms/step

Input: Apparel Ethnic Wear Red Summer
Predicted Fashion Type: Pullover
Confidence: 38.13%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step

Input: Apparel Topwear Black Winter
Predicted Fashion Type: Jacket
Confidence: 29.43%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step

Input: Accessories Watches Blue Casual
Predicted Fashion Type: Cardigan
Confidence: 33.59%
